## Without web search

In [12]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from azure_openai_llm import create_azure_llm
from langchain.agents import create_agent
from langchain.messages import HumanMessage  

llm = create_azure_llm()

agent = create_agent(
    model=llm
)

In [6]:
agent.invoke(
    {"messages": [HumanMessage(content="who is current mayor of lahore?")]}
)['messages'][-1].content

'As of my last update in October 2023, the current mayor of Lahore is Mubashir Javed. He took office in 2023. However, please verify with up-to-date sources, as political positions can change.'

In [9]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)
response['messages'][-1].content

'My training knowledge is up to date until October 2023. If you have any questions or need information within that timeframe, feel free to ask!'

## Add web search tool

In [14]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of Lahore?")

{'query': 'Who is the current mayor of Lahore?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.mayors.asia/city/lahore/',
   'title': 'Lahore | Asian Mayors Forum',
   'content': 'Mayor: Mubesher Javed. Website. Lahore is the capital and largest city of the Pakistani province of Punjab. Lahore',
   'score': 0.99957937,
   'raw_content': None},
  {'url': 'https://en.wikipedia.org/wiki/Mayor_of_Lahore',
   'title': 'Mayor of Lahore - Wikipedia',
   'content': 'Mayors of Lahore ; Mian Amir Mehmood, 2005, 2009 ; Administrator system implemented 2010–2016 ; Mubashar Javed, 2016, 2021 ; Administrator system implemented',
   'score': 0.99933845,
   'raw_content': None},
  {'url': 'https://www.facebook.com/Lord.Mayor.Lahore/',
   'title': 'Lord Mayor Lahore - Facebook',
   'content': 'Samiullah Chaudhary is with M Qasim Khalifah in Lahore, Pakistan. Feb 19, 2016\U000f078b\U000f17e0. Today meeting with my Reverent Quaid Chief Minister Mian Shehb

In [16]:
agent = create_agent(
    llm,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of Lahore?")

response = agent.invoke(
    {"messages": [question]}
)

In [17]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of Lahore?', additional_kwargs={}, response_metadata={}, id='698807b9-e735-4b78-b086-ba82970cedc6'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 51, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DFakcsiBvkdBmoi9vq59TNv11vaVb', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe

In [18]:
pprint(response["messages"][1].tool_calls)

[{'args': {'query': 'current mayor of Lahore 2023'},
  'id': 'call_QuVuumG99KCfLycT0JSuXhbB',
  'name': 'web_search',
  'type': 'tool_call'}]


In [22]:
from langchain_core.tracers.context import tracing_v2_enabled

with tracing_v2_enabled() as ctx:
    question = HumanMessage(content="How is the weather today in Lahore?")

    response = agent.invoke(
        {"messages": [question]}
    )
    pprint(response['messages'][-1].content)
    url = ctx.get_run_url()
    print(f"View the trace at: {url}")

('Today in Lahore, the weather is as follows:\n'
 '\n'
 '- **Temperature**: 30.1°C (86.2°F)\n'
 '- **Condition**: Mist\n'
 '- **Wind**: 6.9 mph (11.2 kph) from the WNW\n'
 '- **Humidity**: 31%\n'
 '- **Visibility**: 4 km\n'
 '- **Feels Like**: 27.9°C (82.2°F)\n'
 '\n'
 'For more details, you can check the [weather update '
 'here](https://www.weatherapi.com/).')
View the trace at: https://eu.smith.langchain.com/o/ed681c67-0bab-40c3-ad5a-fe23a6c3ddb4/projects/p/c01085aa-1fe5-450f-995e-347c849660ab/r/019cb7c2-f732-7481-bab3-62585dc1cfd8?poll=true
